# Writing Bake-off — Round 3: legacy models

Testing whether older models still write better than the current ones.

Four configs, two briefs, 16 calls.

**No blind, no grader, no scores.csv.** When the brief names every banned word, the machine
columns carry little signal, and grading from excerpts can produce a wrong call that only
reading the full text corrects. So this notebook generates and dumps. Judging happens by reading.

**opus-5-med is the control.** It is not here to compete; it is the bridge that makes
round 3 comparable to round 2. Without a shared config across runs, a win here means
nothing next to the earlier numbers.

**A confound worth naming.** A preference formed over months of real work differs from one
cold prompt. If an older model loses here, that is evidence about cold-prompt writing, not
about a preference formed in real use.

**Fill the placeholders first.** The briefs contain `<...>` placeholders and `[Placeholder ...]`
markers. Replace them with your own material before you run.

**Pricing note.** Opus 4.6, 4.7, 4.8 and 5 all bill at $5/$25. If 4.6 wins, the swap is
free. And Claude 4.7 and later tokenize the same text into roughly 30% more tokens, so
4.6 is quietly cheaper per word than Opus 5 at the same rate.

## 1. Setup

In [ ]:
import os, sys, subprocess, json, re, html, time
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    import httpx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=True)
    import httpx

VOLUME  = Path("./local-data")
ROOT    = VOLUME / "bakeoff_r3_legacy"
SOURCES = VOLUME / "writing_bakeoff" / "sources"     # reuse round 2's docs if present
RAW     = ROOT / "raw_results.jsonl"
READING = ROOT / "READING_PACK.md"

ROOT.mkdir(parents=True, exist_ok=True)
SOURCES.mkdir(parents=True, exist_ok=True)

if not os.environ.get("OPENROUTER_API_KEY", "").strip():
    print("OPENROUTER_API_KEY not set. Attach the secret in the sidebar, or:\n"
          "  os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-...'")
else:
    print(f"ready -> {ROOT}")

## 2. Config

Legacy models run at **default effort**, no reasoning parameter. The question is whether
an older model still writes better, not how it responds to a dial that was not part of the
setup where the preference formed.

`anthropic/claude-3-opus` may be retired; it does not appear on any of the current
provider listings. The probe will tell you. If it fails, delete that entry and move on.

In [ ]:
MODELS = {
    # The legacy models under test, at default.
    "opus-4.6":   {"slug": "anthropic/claude-opus-4.6",   "effort": None},
    "sonnet-4.6": {"slug": "anthropic/claude-sonnet-4.6", "effort": None},
    "opus-3":     {"slug": "anthropic/claude-3-opus",     "effort": None},

    # Control, carried over from an earlier round; makes the two runs comparable.
    "opus-5-med": {"slug": "anthropic/claude-opus-5",     "effort": "medium"},
}

# If opus-4.6 404s, OpenRouter's own migration cookbook uses the reversed form
# "anthropic/claude-4.6-opus". Try that before assuming the model is gone.

# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's confirmed listed price in
# USD per million tokens (input, output). A model with no confirmed price is left out of PRICE, and the notebook
# then refuses to run before any paid call.
import math
import os

# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent
from datetime import datetime, timezone


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # one cap per run; override with the RUN_BUDGET_USD environment variable
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed listed prices, USD per million tokens (input, output). Source for each entry:
# https://openrouter.ai/api/v1/models/<slug>/endpoints, retrieved 2026-11-05, highest price across the listed endpoints.
PRICE = {
    "anthropic/claude-opus-4.6":   (5.5, 27.5),
    "anthropic/claude-sonnet-4.6": (3.3, 16.5),
    "anthropic/claude-opus-5":     (10.0, 50.0),
    # anthropic/claude-3-opus: no listed price could be confirmed on a primary source, so it is left out on purpose.
}
missing_prices = sorted(set([m["slug"] for m in MODELS.values()]) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"No confirmed price for {missing_prices}, so this notebook will not run. Confirm the listed "
                       "price (USD per million tokens, input and output) and add it to PRICE in this cell, or delete "
                       "that entry from MODELS.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
# One ledger and lock root shared by every notebook: SPEND_LEDGER_DIR, else <repo root>/spend_ledger.
# Repo root = first directory holding .git walking up from the working directory (no .git: SPEND_LEDGER_DIR is required).
LEDGER_ROOT = _ledger_root()
os.makedirs(LEDGER_ROOT, exist_ok=True)
RUN_DIR = os.path.join(LEDGER_ROOT, RUN_ID)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One paid runner at a time across every notebook. One exclusive lock file at LEDGER_ROOT/runner.lock: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died
# without exiting) is never cleared automatically: confirm no runner is alive, then delete LEDGER_ROOT/runner.lock
# by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using this run, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)
os.makedirs(RUN_DIR, exist_ok=True)


REPS       = {"B-shortform": 3, "A-longform": 1}
MAX_TOKENS = 32000
TIMEOUT_S  = 900
WORKERS    = 8
RETRIES    = 3

TOTAL = sum(REPS[b] for _ in MODELS for b in REPS)
print(f"{len(MODELS)} configs, {TOTAL} calls")

## 3. Sources and briefs

In [ ]:
FR_DOCS = {
    "fsis_salmonella_initiative_2026-14216.txt":
        "https://www.federalregister.gov/documents/full_text/text/2026/07/15/2026-14216.txt",
    "fda_citrus_red_2_revocation_2026-14909.txt":
        "https://www.federalregister.gov/documents/full_text/text/2026/07/23/2026-14909.txt",
}

def clean(t):
    t = re.sub(r"(?s)<head>.*?</head>", "", t)
    t = re.sub(r"<[^>]+>", "", t)
    return re.sub(r"\n{3,}", "\n\n", html.unescape(t)).strip()

for name, url in FR_DOCS.items():
    p = SOURCES / name
    if not p.exists():
        p.write_text(clean(httpx.get(url, timeout=90).text), encoding="utf-8")
    print(f"  {name}: {len(p.read_text().split())} words")

SOURCE_TEXT = "\n\n---\n\n".join(
    f"[SOURCE FILE: {p.name}]\n\n{p.read_text(encoding='utf-8')}"
    for p in sorted(SOURCES.glob("*.txt"))
)

In [ ]:
BRIEF_A = """You are writing a regulatory intelligence report for a client of a regulatory consultancy. The client is an industry association. The report goes out under the analyst's name and gets read by the client's executive director and passed to its member companies.

Write the report from the source material below. Do not use anything outside it.

## What the report has to do

Explain what happened, what it requires of member companies, and what to watch next. The reader knows food safety; they do not know this specific document. They need to finish reading and know what to do on Monday.

Length: 2,000 words or more.

## Structure

Open with what happened and why the reader is getting this. No preamble about the importance of food safety.

Then the body, organized however the material actually organizes itself. Do not force a template onto it.

Close with what to watch next. Specific things: dates, comment periods, follow-on actions, named agencies.

## Citation

Every factual claim traces to the source material below. Cite inline by document name and section or paragraph, like: (FSIS Recall 012-2026, para. 4).

If the source does not answer a question the report needs answered, write that the source does not address it. Do not fill the gap with what is usually true. An analyst who invents a number to make a report feel complete is worse than one who leaves a hole marked.

## Voice

Plain and direct. Sentences under 16 words where the meaning allows it.

Do not use: pivotal, crucial, enhance, underscore, landscape, vibrant, testament, foster, delve, showcase, intricate, evolving, cultivate, crafting, utilize, execute as a verb.

No contrast structures. No "it's not X, it's Y." No "despite this." No "but here's the thing."

No sentence that tells the reader something is important. Show what it requires and let them judge.

No summary ending. No "in conclusion," no "overall," no wrap-up paragraph that restates the report.

No vague authority. No "experts say," no "industry reports suggest." Name the source or cut the claim.

Semicolons rather than em-dashes.

Any statistic gets a date and a named source, or it comes out.

Do not quote the source unless the exact wording matters; when you do quote, quote it correctly.

## Output

The report only. No cover note, no explanation of your approach, no offer to revise.

---

# SOURCE MATERIAL

""" + SOURCE_TEXT

BRIEF_B = """Write a LinkedIn post as the persona described below.

## Who you are

You are a professional writer with ten years of experience in B2B software. [Placeholder persona: replace with your own.]

## What to write about

He has run a benchmark on which AI model to use for regulatory work. <your benchmark: how many calls, across how many models>. The finding that surprised him: <your headline finding>. <two or three supporting numbers from your benchmark, each with its unit>.

<your own cost comparison, for example the cheapest and the most expensive model you tested and where each landed; or delete this paragraph>

His conclusion: <your conclusion, in one or two sentences>. And a human still reads everything before it ships.

## Shape

Hook, then receipts, then principle. That order.

The hook is a real observation, not a question and not a statistic dressed as a hook. No "Here's what I learned." No "I ran an experiment. The results shocked me."

The receipts are the numbers above. Use them. They are his own and they are dated <date of your benchmark>.

The principle lands at the end and does not preach. Write it in your own words; do not reuse the phrasing above.

Length: 150 to 250 words.

## Voice rules

Write in lowercase. Do not use sentence case.

No hashtags. No engagement bait; no "thoughts?", no "agree?", no "drop a comment."

Semicolons rather than em-dashes.

Sentences under 16 words.

Vary the rhythm. Do not put every sentence on its own line; uniform one-line paragraphs read as machine-generated even when each line is fine.

He does not say "folks," "quick one," "biz," "folks in the space," or "let's dive in." He does not sound like a tech bro.

Do not use: pivotal, crucial, enhance, underscore, landscape, vibrant, testament, foster, delve, showcase, intricate, evolving, cultivate, crafting, utilize, execute as a verb.

No contrast structures. No "it's not X, it's Y." No "the ceiling isn't the tool, it's the task." No "not A. Not B. But C." No "despite this."

No cringe transition questions. No "The catch?" No "The kicker?" No "Sound familiar?"

No invented people. No "a QA manager I know named Pat."

No importance sentences. Nothing about what this means for the broader industry.

No summary ending.

Every fact in this post comes from the benchmark described above. Do not invent additional specifics: no dollar amounts, durations, or personal anecdotes that are not stated here.

## Output

The post only. No caption, no alternate versions, no explanation."""

BRIEFS = {"A-longform": BRIEF_A, "B-shortform": BRIEF_B}
print({k: f"{len(v.split())} words" for k, v in BRIEFS.items()})

## 4. Probe

Four short calls. `anthropic/claude-3-opus` is the one most likely to fail.

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=300, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = httpx.post(URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


URL = "https://openrouter.ai/api/v1/chat/completions"

def headers():
    return {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://example.com/labs-benchmark",
            "X-Title": "labs-benchmark"}

def build(cfg, messages, max_tokens):
    m = MODELS[cfg]
    p = {"model": m["slug"], "messages": messages, "max_tokens": max_tokens,
         "temperature": 1.0, "usage": {"include": True}}
    if m["effort"]:
        p["reasoning"] = {"effort": m["effort"]}
    return p

def cost_of(cfg, ptok, ctok):
    pin, pout = PRICE[MODELS[cfg]["slug"]]
    if ptok is None or ctok is None:
        return None
    return ptok / 1e6 * pin + ctok / 1e6 * pout

def call(cfg, messages, max_tokens=None):
    mt = max_tokens or MAX_TOKENS
    p = build(cfg, messages, mt)
    last = None
    for i in range(RETRIES):
        t0 = time.time()
        try:
            r = guarded_post(p, headers(), timeout=TIMEOUT_S, purpose=cfg)
            el = round(time.time() - t0, 2)
            if r.status_code in (400, 404):
                return {"ok": False, "error": f"{r.status_code} {r.text[:300]}", "latency_s": el}
            r.raise_for_status()
            d = r.json()
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300], "latency_s": el}
            ch = d["choices"][0]
            text = ch["message"].get("content")
            u = d.get("usage") or {}
            if not text or not text.strip():
                last = f"empty content (finish={ch.get('finish_reason')})"
                if i < RETRIES - 1:
                    mt = int(mt * 1.5); p = build(cfg, messages, mt); continue
                return {"ok": False, "error": last, "latency_s": el}
            return {"ok": True, "text": text, "latency_s": el,
                    "finish_reason": ch.get("finish_reason"),
                    "prompt_tokens": u.get("prompt_tokens"),
                    "completion_tokens": u.get("completion_tokens"),
                    "reasoning_tokens": (u.get("completion_tokens_details") or {}).get("reasoning_tokens"),
                    "cost_computed": cost_of(cfg, u.get("prompt_tokens"), u.get("completion_tokens"))}
        except BudgetExceeded:
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            if i < RETRIES - 1:
                time.sleep(2 ** i)
    return {"ok": False, "error": last or "unknown", "latency_s": None}


bad = []
for name in list(MODELS):
    res = call(name, [{"role": "user", "content": "Reply with the single word: ok"}], 2000)
    if res["ok"]:
        print(f"  OK    {name:12s} {MODELS[name]['slug']:32s} {res['latency_s']:>6}s")
    else:
        bad.append(name)
        print(f"  FAIL  {name:12s} {MODELS[name]['slug']:32s} {res['error'][:110]}")

if bad:
    print(f"\n{len(bad)} failed: {', '.join(bad)}")
    print("Fix the slug in MODELS, or drop the entry with:  del MODELS['name']")
    print("Real slugs: https://openrouter.ai/models")
else:
    print("\nAll four resolved.")

## 5. Run

In [ ]:
done = set()
if RAW.exists():
    for line in RAW.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            if r.get("ok") and isinstance(r.get("text"), str) and r["text"].strip():
                done.add((r["model"], r["brief"], r.get("rep", 1)))

jobs = [(m, b, rep) for m in MODELS for b in BRIEFS
        for rep in range(1, REPS[b] + 1) if (m, b, rep) not in done]
print(f"{len(jobs)} to run, {len(done)} done.\n")

def work(j):
    m, b, rep = j
    res = call(m, [{"role": "user", "content": BRIEFS[b]}])
    res.update({"model": m, "brief": b, "rep": rep, "slug": MODELS[m]["slug"],
                "effort": MODELS[m]["effort"], "ts": datetime.now(timezone.utc).isoformat()})
    return res

if jobs:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = [pool.submit(work, j) for j in jobs]
        with RAW.open("a", encoding="utf-8") as fh:
            for f in as_completed(futs):
                r = f.result()
                fh.write(json.dumps(r, ensure_ascii=False) + "\n"); fh.flush()
                tag = f"{r['model']}#{r['rep']}"
                if r["ok"]:
                    print(f"  done  {tag:16s} {r['brief']:12s} {len(r['text'].split()):>5}w  "
                          f"{r['latency_s']:>6}s  ${r.get('cost_computed') or 0:.4f}")
                else:
                    print(f"  FAIL  {tag:16s} {r['brief']:12s} {r['error'][:70]}")

## 6. Reading pack

Writes one markdown file with every output in full, model names attached, plus a metrics
table. Open it, or paste it into the chat for judging.

Short-form comes first because it is 16 posts of a couple hundred words each and reads on
a phone; long-form is four reports and wants a laptop.

In [ ]:
recs = []
for line in RAW.read_text(encoding="utf-8").splitlines():
    if line.strip():
        r = json.loads(line)
        if r.get("ok") and isinstance(r.get("text"), str) and r["text"].strip():
            recs.append(r)
latest = {(r["model"], r["brief"], r.get("rep", 1)): r for r in recs}
recs = list(latest.values())

out = ["# Round 3 reading pack — legacy models", "",
       f"Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC. "
       f"{len(recs)} of {TOTAL} outputs.", "",
       "opus-5-med is the control, repeated so rounds are comparable.", "", "## Metrics", "",
       "| config | brief | rep | words | latency | $ | ctok | rtok |",
       "|---|---|---|---|---|---|---|---|"]

for r in sorted(recs, key=lambda x: (x["brief"], x["model"], x.get("rep", 1))):
    out.append(f"| {r['model']} | {r['brief']} | {r.get('rep',1)} | {len(r['text'].split())} | "
               f"{r['latency_s']}s | ${r.get('cost_computed') or 0:.4f} | "
               f"{r.get('completion_tokens')} | {r.get('reasoning_tokens')} |")

for brief, title in [("B-shortform", "Short-form posts"), ("A-longform", "Long-form reports")]:
    out += ["", f"## {title}", ""]
    for r in sorted([x for x in recs if x["brief"] == brief],
                    key=lambda x: (x["model"], x.get("rep", 1))):
        out += [f"### {r['model']} — rep {r.get('rep',1)}  ({len(r['text'].split())} words)", "",
                r["text"].strip(), "", "---", ""]

READING.write_text("\n".join(out), encoding="utf-8")
print(f"{len(recs)} outputs -> {READING}")
print(f"{READING.stat().st_size/1024:.0f} KB")
print("\nShort-form section alone is small enough to paste into chat.")